# Fine-tuning di TrackNet sul tennis (Colab, GPU)

Questo notebook **riaddestra un po'** (fine-tuning) la rete della pallina, TrackNet, partendo dai pesi che usa già il programma (`TrackNet_best.pt`, addestrati sul **badminton**) e mostrandole le palline da **tennis** del dataset pubblico del primo TrackNet (riprese TV di 10 partite, 1280x720). Poi dà il **voto** prima/dopo sul **set di test etichettato a mano** (video mai usati per addestrare).

Il programma principale **non cambia**: i nuovi pesi finiscono su Drive in `tracknet_finetune/<RUN>/TrackNet_tennis.pt`. Usarli nel programma è il passo successivo (integrazione), da fare solo se il voto migliora.

**Preparazione:** niente. I pesi di partenza (`ckpts/TrackNet_best.pt`) sono già su Drive; il dataset pubblico lo copia e lo scarica la cella 5; il set di test etichettato arriva da solo su Drive (`tracknet_finetune/etichette_test.csv`) quando etichetti sul PC.

**Come si usa:** Runtime → Cambia tipo di runtime → **GPU A100** (va anche la T4, più lenta), poi **Runtime → Esegui tutto**. Servono solo due autorizzazioni di Google: il Drive (cella 2) e, la prima volta, la copia del dataset (cella 5).
- Se la **prova** (cella 7) non riesce, il notebook si ferma lì e l'addestramento non parte: mandami l'errore.
- L'**addestramento** (cella 8) dura indicativamente 30-60 minuti su A100; i risultati vanno su Drive in `tracknet_finetune/run1/`.
- Lo **sguardo veloce** (cella 4b) è facoltativo e non chiede etichette: si può lanciare subito dopo le celle 1-4, per vedere presto se i pesi nuovi sono chiaramente peggio. Non è il voto.
- Il **voto** (cella 10) funziona solo quando il set di test è etichettato; prima si ferma con un messaggio, ed è normale.

Se Colab si disconnette durante l'addestramento: riapri ed "Esegui tutto": il dataset pronto torna dal Drive in pochi minuti e la cella 8 riprende dall'ultima epoca salvata.

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Codice: il nostro da GitHub e quello originale di TrackNetV3

Il codice di addestramento è quello **originale** di TrackNetV3 (qaz812345/TrackNetV3, licenza MIT), fermo alla versione verificata. La cella controlla che il modello sia identico a `tracknet3/model.py` del programma: così i pesi nuovi si usano senza cambiare niente.

In [ ]:
import os, shutil, glob, subprocess, sys, json

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"
FT_DRIVE = os.path.join(DRIVE_DIR, "tracknet_finetune")      # risultati del fine-tuning su Drive
TNV3_DIR = "/content/TrackNetV3"                               # codice originale di addestramento
TNV3_COMMIT = "6eda442ada1740573f200f836d93edc9a541ee86"       # versione verificata

# Il repo e' pubblico: il token serve solo se un giorno torna privato (come nel notebook principale).
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
assert os.path.isdir(os.path.join(WORK_DIR, "tracknet_finetune")), "Su GitHub manca tracknet_finetune/: fai prima il push dal PC"

if not os.path.isdir(TNV3_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/qaz812345/TrackNetV3.git", TNV3_DIR], check=True)
subprocess.run(["git", "-C", TNV3_DIR, "checkout", "-q", TNV3_COMMIT], check=True)

leggi = lambda p: open(p, encoding="utf-8").read().replace("\r\n", "\n")
assert leggi(os.path.join(TNV3_DIR, "model.py")) == leggi(os.path.join(WORK_DIR, "tracknet3", "model.py")), \
    "Il modello di TrackNetV3 non e' identico a tracknet3/model.py: non proseguire"
print("codice pronto; modello identico a quello del programma")

In [ ]:
!pip install -q parse pycocotools gdown

## 4. Pesi di partenza (quelli del badminton che usa il programma)

In [ ]:
PESI_INIZIALI = os.path.join(WORK_DIR, "tracknet3", "ckpts", "TrackNet_best.pt")
os.makedirs(os.path.dirname(PESI_INIZIALI), exist_ok=True)
shutil.copy2(os.path.join(DRIVE_DIR, "ckpts", "TrackNet_best.pt"), PESI_INIZIALI)   # copy2: tiene la data del file
print("pesi di partenza:", PESI_INIZIALI, f"({os.path.getsize(PESI_INIZIALI) / 1e6:.0f} MB)")

## 4b. Sguardo veloce (facoltativo, senza etichette)

Si può lanciare **subito dopo le celle 1-4**, prima di etichettare. Confronta i pesi del badminton e quelli nuovi (`RUN`) sui nostri video, nei fotogrammi attorno ai colpi trovati dal programma: quante volte TrackNet vede la pallina, salti sospetti, punti isolati, e ritagli con i punti sovrapposti (badminton = cerchio giallo, nuovi = quadrato azzurro).

**Non è il voto**: senza etichette un punto falso conta come "visto". Serve a capire presto se i pesi nuovi sono chiaramente peggio. Il voto vero è la cella 10. Si salta da solo se i pesi nuovi non ci sono (`ESEGUI_SGUARDO = False` per saltarlo a mano). Le previsioni vanno su Drive (`tracknet_finetune/valutazione/previsioni`) e la cella 10 le riusa.

In [ ]:
# TrackNet (predict.py) importa 'parse': se la cella 3 con il pip install non e' stata eseguita, lo installo qui
try:
    import parse  # noqa: F401
except ImportError:
    !pip install -q parse pycocotools

ESEGUI_SGUARDO = True          # False per saltare lo sguardo veloce
SGUARDO_VIDEO = ["nicola_matarese_trim.mp4", "zverev_djokovic_trim_swin_like.mp4",
                 "alcaraz.mp4"]

from IPython.display import Markdown, Image, display

RUN_S = globals().get("RUN", "run1")
PESI_S = {"badminton": PESI_INIZIALI, RUN_S: os.path.join(FT_DRIVE, RUN_S, "TrackNet_tennis.pt")}
VAL_LOCALE, VAL_DRIVE = "/content/valutazione", os.path.join(FT_DRIVE, "valutazione")
SCRIPT = os.path.join(WORK_DIR, "tracknet_finetune", "sguardo_veloce.py")

if not ESEGUI_SGUARDO:
    print("Sguardo veloce saltato (ESEGUI_SGUARDO = False).")
elif not os.path.exists(PESI_S[RUN_S]):
    print(f"Sguardo veloce saltato: i pesi '{RUN_S}' non sono ancora addestrati.")
else:
    if not os.path.exists(SCRIPT):                       # non ancora su GitHub: lo prendo dal Drive
        src = os.path.join(FT_DRIVE, "sguardo_veloce.py")
        if not os.path.exists(src):
            raise SystemExit("Manca sguardo_veloce.py (ne' su GitHub ne' su Drive in tracknet_finetune/).")
        shutil.copy(src, SCRIPT)
    os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
    for v in SGUARDO_VIDEO:
        src, dst = os.path.join(DRIVE_DIR, "inputs", v), os.path.join(WORK_DIR, "inputs", v)
        if not os.path.exists(src):
            print("  manca su Drive inputs/:", v)
        elif not os.path.exists(dst):
            shutil.copy(src, dst)
    if os.path.isdir(os.path.join(VAL_DRIVE, "previsioni")):          # previsioni gia' fatte: si riusano
        shutil.copytree(os.path.join(VAL_DRIVE, "previsioni"), os.path.join(VAL_LOCALE, "previsioni"), dirs_exist_ok=True)
    pesi_arg = " ".join(f'"{n}={p}"' for n, p in PESI_S.items())
    dati_dir = os.path.join(DRIVE_DIR, "outputs", "dati")
    %cd {WORK_DIR}
    !python tracknet_finetune/sguardo_veloce.py --video_dir inputs --dati_dir "{dati_dir}" --pesi {pesi_arg} --uscita {VAL_LOCALE}
    os.makedirs(VAL_DRIVE, exist_ok=True)
    for sotto in ("previsioni", "sguardo"):
        if os.path.isdir(os.path.join(VAL_LOCALE, sotto)):
            shutil.copytree(os.path.join(VAL_LOCALE, sotto), os.path.join(VAL_DRIVE, sotto), dirs_exist_ok=True)
    md = os.path.join(VAL_LOCALE, "sguardo", "sguardo.md")
    if os.path.exists(md):
        display(Markdown(open(md, encoding="utf-8").read()))
    for f in sorted(glob.glob(os.path.join(VAL_LOCALE, "sguardo", "*.png"))):
        print(os.path.basename(f))
        display(Image(filename=f))
    print("Copia dei risultati su Drive:", VAL_DRIVE)

## 5. Dataset pubblico del tennis (TrackNet 2019)

Il dataset è il file `Dataset.zip` nella [cartella condivisa](https://drive.google.com/drive/folders/11r0RUaQHX7I3ANkaYG4jOxXK1OYo01Ut) del README di yastrebksv/TrackNet. Google ne blocca spesso il download diretto ("Too many users have viewed or downloaded this file"), quindi la cella:

1. ne fa una **copia nel tuo Drive** (`Il mio Drive/Dataset_tennis_TrackNet.zip`): la copia la fa Google sui suoi server e il limite non vale. La prima volta compare una finestra di **autorizzazione** di Google: accetta;
2. scarica la copia qui e la scompatta (qualche minuto).

Le volte dopo usa la copia già nel tuo Drive, e se il dataset è già stato preparato (cella 6, copia su Drive) salta tutto.

In [ ]:
import re
MAX_FRAME = 10000          # fotogrammi di addestramento (circa), usati dalla cella 6
VAL_PARTITE = "9,10"       # partite usate solo per la validazione
DATI = "/content/dati_tennis"
tar_drive = os.path.join(FT_DRIVE, f"dati_tennis_{MAX_FRAME}_val{VAL_PARTITE.replace(',', '-')}.tar")

DATASET_ID = "11r0RUaQHX7I3ANkaYG4jOxXK1OYo01Ut"      # cartella condivisa del dataset
ZIP_ID = "1DQ3ZbvokTsgOq6x-ay6O8U2W4a8e3LFw"          # Dataset.zip in quella cartella
NOME_COPIA = "Dataset_tennis_TrackNet.zip"            # la copia nel tuo Drive
NOMI_ZIP = {"Dataset.zip", "Copia di Dataset.zip", "Copy of Dataset.zip", NOME_COPIA}
RAW = "/content/tennis_raw"

def label_in(cartella):
    return glob.glob(os.path.join(cartella, "**", "Label.csv"), recursive=True)

def radice_partite(cartella):
    """La cartella che contiene game1, game2, ... (Label.csv sta in <radice>/gameN/ClipM/)."""
    et = label_in(cartella)
    return os.path.dirname(os.path.dirname(os.path.dirname(et[0]))) if et else None

def scompatta(z):
    print("scompatto", os.path.basename(z), "...")
    os.makedirs(RAW, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", z, "-d", RAW], check=True)

def copia_e_scarica():
    """Copia Dataset.zip nel tuo Drive (copia fatta da Google, niente limite di download) e la scarica qui."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    srv = build("drive", "v3")
    gia = srv.files().list(q=f"name = '{NOME_COPIA}' and trashed = false and 'me' in owners",
                           fields="files(id)").execute().get("files", [])
    if gia:
        copia_id = gia[0]["id"]
        print("copia gia' nel tuo Drive:", NOME_COPIA)
    else:
        print(f"copio Dataset.zip nel tuo Drive (Il mio Drive/{NOME_COPIA}) ...")
        copia_id = srv.files().copy(fileId=ZIP_ID, body={"name": NOME_COPIA}, supportsAllDrives=True,
                                    fields="id").execute()["id"]
    locale = os.path.join("/content", NOME_COPIA)
    print("scarico la copia ...")
    with open(locale, "wb") as fh:
        dl = MediaIoBaseDownload(fh, srv.files().get_media(fileId=copia_id), chunksize=200 * 1024 * 1024)
        fatto = False
        while not fatto:
            stato, fatto = dl.next_chunk()
            if stato:
                print(f"  {stato.progress() * 100:.0f}%")
    scompatta(locale)
    os.remove(locale)

ORIGINE = None
if os.path.exists(os.path.join(DATI, "riepilogo_dataset.json")) or os.path.exists(tar_drive):
    print("Dataset gia' preparato (in questa sessione o su Drive): la cella 6 lo usa, qui non serve altro.")
else:
    ORIGINE = radice_partite(RAW)                                        # gia' scompattato in questa sessione
    if ORIGINE is None:                                                  # zip gia' nel tuo Drive
        zip_drive = [z for z in glob.glob("/content/drive/MyDrive/*.zip")
                     + glob.glob(os.path.join(DRIVE_DIR, "datasets", "**", "*.zip"), recursive=True)
                     if os.path.basename(z) in NOMI_ZIP]
        if zip_drive:
            scompatta(zip_drive[0])
            ORIGINE = radice_partite(RAW)
    if ORIGINE is None:                                                  # copia nel tuo Drive e scarica
        try:
            copia_e_scarica()
            ORIGINE = radice_partite(RAW)
        except Exception as e:
            print("Copia automatica non riuscita:", str(e)[:400])
    if ORIGINE is None:
        raise SystemExit("Dataset NON trovato. Mandami in chat il messaggio qui sopra. Strada a mano: apri "
                         f"https://drive.google.com/drive/folders/{DATASET_ID} , clic destro su Dataset.zip -> "
                         "Crea una copia, poi rilancia questa cella.")
    clip = label_in(ORIGINE)
    partite = sorted({int(m.group(1)) for c in clip
                      for m in [re.search(r"game\s*(\d+)", os.path.relpath(c, ORIGINE), re.IGNORECASE)] if m})
    print(f"Dataset in {ORIGINE}: {len(clip)} clip, partite {partite}")
    if len(partite) < 10:
        print("ATTENZIONE: mancano delle partite (zip incompleto?). Il dataset completo ne ha 10.")

## 6. Prepara il dataset per TrackNetV3

Prende circa `MAX_FRAME` fotogrammi per l'addestramento (impostato nella cella 5) e tiene le partite `VAL_PARTITE` **solo** per la validazione (divisione per partita, non per fotogramma). Salva i fotogrammi già a 512x288. Con `SALVA_SU_DRIVE = True` mette una copia del risultato su Drive (circa 1,7 GB): la volta dopo, o dopo una disconnessione, si riparte da lì in pochi minuti.

In [ ]:
SALVA_SU_DRIVE = True

os.makedirs(FT_DRIVE, exist_ok=True)
%cd {WORK_DIR}
if os.path.exists(os.path.join(DATI, "riepilogo_dataset.json")):
    print("dataset gia' pronto in", DATI)
elif os.path.exists(tar_drive):
    print("recupero il dataset pronto dal Drive:", tar_drive)
    subprocess.run(["tar", "-xf", tar_drive, "-C", "/content"], check=True)
else:
    if not ORIGINE:
        raise SystemExit("Manca il dataset pubblico: vedi la cella 5")
    if os.path.isdir(DATI):
        shutil.rmtree(DATI)                      # resto di un tentativo interrotto
    !python tracknet_finetune/prepara_dataset_tennis.py --origine "{ORIGINE}" --uscita {DATI} --max_frame {MAX_FRAME} --val_partite {VAL_PARTITE}
    if not os.path.exists(os.path.join(DATI, "riepilogo_dataset.json")):
        raise SystemExit("Preparazione non riuscita: guarda l'errore qui sopra e mandamelo in chat")
    if SALVA_SU_DRIVE:
        subprocess.run(["tar", "-cf", tar_drive, "-C", "/content", "dati_tennis"], check=True)
        print("copia salvata su Drive:", tar_drive)

riepilogo = json.load(open(os.path.join(DATI, "riepilogo_dataset.json")))
for split, s in riepilogo["split"].items():
    print(f"{split}: {s['fotogrammi']} fotogrammi in {len(s['clip'])} clip, pallina visibile {s['visibili']}, mossa {s['mosse']}")

## 7. Prova veloce (2-3 minuti)

256 campioni, 1 epoca: serve solo a scoprire subito se qualcosa non va (dati, memoria, versioni). Deve finire con `PROVA RIUSCITA`; se non riesce, la cella si ferma e con "Esegui tutto" l'addestramento non parte. Scrive in una cartella temporanea, non sul Drive.

In [ ]:
%cd {WORK_DIR}
shutil.rmtree("/content/prova_ft", ignore_errors=True)
!python tracknet_finetune/addestra.py --dati {DATI} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita /content/prova_ft --prova
if not os.path.exists("/content/prova_ft/ultimo_completo.pt"):
    raise SystemExit("La prova NON e' riuscita: copia l'errore qui sopra e mandamelo in chat.")
print("\nProva riuscita: l'addestramento (cella 8) puo' partire.")

## 8. Addestramento

Prima misura i pesi del badminton sulla validazione (il "prima"), poi addestra per `EPOCHE` epoche e dopo ognuna rimisura: tiene la versione migliore. Tutto va su Drive in `tracknet_finetune/<RUN>/`:
- `TrackNet_tennis.pt` (la migliore: è quella da usare),
- `ultimo_completo.pt` (per riprendere),
- `registro.json` (i numeri di ogni epoca).

Per una seconda prova con altre impostazioni cambia `RUN` (es. `run2`): la prima resta.

La cella decide da sola: se `<RUN>` è già finito **salta** (così "Esegui tutto" arriva fino al voto); se era stato interrotto (Colab disconnesso) **riprende** dall'ultima epoca salvata; altrimenti parte da zero.

In [ ]:
RUN = "run1"          # nome della prova
EPOCHE = 15
LR = 1e-4             # passo piccolo: si corregge la rete del badminton, non si riparte da zero
CONGELA = "nessuno"   # "inizio" = tiene fermi i primi blocchi (se la validazione peggiora presto)

USCITA = os.path.join(FT_DRIVE, RUN)
p_reg = os.path.join(USCITA, "registro.json")
fatte = len(json.load(open(p_reg))["epoche"]) if os.path.exists(p_reg) else 0
if fatte >= EPOCHE and os.path.exists(os.path.join(USCITA, "TrackNet_tennis.pt")):
    print(f"{RUN}: addestramento gia' finito ({fatte} epoche), salto.")
else:
    opz_riprendi = ""
    if os.path.exists(os.path.join(USCITA, "ultimo_completo.pt")):
        opz_riprendi = "--riprendi"
        print(f"{RUN}: riprendo dall'epoca {fatte + 1}")
    %cd {WORK_DIR}
    !python tracknet_finetune/addestra.py --dati {DATI} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita "{USCITA}" --epoche {EPOCHE} --lr {LR} --congela {CONGELA} {opz_riprendi}
fatte = len(json.load(open(p_reg))["epoche"]) if os.path.exists(p_reg) else 0
if fatte < EPOCHE or not os.path.exists(os.path.join(USCITA, "TrackNet_tennis.pt")):
    raise SystemExit(f"Addestramento INTERROTTO dopo {fatte} epoche su {EPOCHE} (lo stato e' salvato su Drive): "
                     "rilancia questa cella per riprendere. Se qui sopra c'e' un errore, mandamelo in chat.")
print(f"\nAddestramento finito ({fatte} epoche). Pesi nuovi:", os.path.join(USCITA, "TrackNet_tennis.pt"))

## 9. Andamento sulla validazione

In [ ]:
import matplotlib.pyplot as plt
reg = json.load(open(os.path.join(USCITA, "registro.json")))
ep = [e["epoca"] for e in reg["epoche"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, nome in (("accuracy", "accuratezza"), ("f1", "F1"), ("recall", "richiamo")):
    ax[0].plot(ep, [e[k] for e in reg["epoche"]], marker="o", label=nome)
    if "prima" in reg:
        ax[0].axhline(reg["prima"][k], ls="--", lw=0.8, color=ax[0].lines[-1].get_color())
ax[0].set_title("validazione (tratteggio = pesi del badminton)"); ax[0].set_xlabel("epoca"); ax[0].legend()
ax[1].plot(ep, [e["perdita_train"] for e in reg["epoche"]], marker="o", label="addestramento")
ax[1].plot(ep, [e["perdita_val"] for e in reg["epoche"]], marker="o", label="validazione")
ax[1].set_title("perdita"); ax[1].set_xlabel("epoca"); ax[1].legend()
plt.show()
if "migliore" in reg:
    print("migliore: epoca", reg["migliore"]["epoca"], "accuratezza", round(reg["migliore"]["accuracy"], 3))

## 10. Voto sul set di test (i nostri video, etichettati a mano)

Confronta i pesi del badminton con quelli nuovi sugli **stessi fotogrammi** etichettati, con la previsione fatta **come nel programma** (`tracknet3/predict.py`, senza InpaintNet). Si può lanciare anche **prima** dell'addestramento: misura solo i pesi del badminton.

Confronta badminton, run1 e run2 (le prove che mancano vengono saltate); per altre prove aggiungi i loro pesi a `PESI`. Le previsioni già fatte restano su Drive (`tracknet_finetune/valutazione/previsioni`) e vengono riusate.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

# I pesi da confrontare sugli stessi fotogrammi. Se una cartella non c'e' ancora, viene saltata.
PESI = {"badminton": PESI_INIZIALI,
        "run1": os.path.join(FT_DRIVE, "run1", "TrackNet_tennis.pt"),
        "run2": os.path.join(FT_DRIVE, "run2", "TrackNet_tennis.pt")}

mancano = [n for n, p in PESI.items() if not os.path.exists(p)]
for n in mancano:
    print(f"(salto '{n}': pesi non ancora addestrati)")
    PESI.pop(n)

et_drive = os.path.join(FT_DRIVE, "etichette_test.csv")
et_repo = os.path.join(WORK_DIR, "tracknet_finetune", "test", "etichette_test.csv")
ETICHETTE = et_drive if os.path.exists(et_drive) else et_repo
if not os.path.exists(ETICHETTE):
    raise SystemExit("Set di test non ancora etichettato: lancia questa cella quando hai finito con etichetta_pallina.py sul PC.")
et = pd.read_csv(ETICHETTE)
print(f"set di test: {ETICHETTE}")
print(f"  {len(et)} etichette: visibile {(et['visibile'] == 1).sum()}, non visibile {(et['visibile'] == 0).sum()}, "
      f"incerta {(et['visibile'] == -1).sum()} (non conta)")

os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
for v in sorted(et["video"].unique()):
    src, dst = os.path.join(DRIVE_DIR, "inputs", v), os.path.join(WORK_DIR, "inputs", v)
    if not os.path.exists(src):
        print("  manca su Drive inputs/:", v, "(le sue etichette vengono saltate)")
    elif not os.path.exists(dst):
        shutil.copy(src, dst)

VAL_LOCALE, VAL_DRIVE = "/content/valutazione", os.path.join(FT_DRIVE, "valutazione")
if os.path.isdir(os.path.join(VAL_DRIVE, "previsioni")):
    shutil.copytree(os.path.join(VAL_DRIVE, "previsioni"), os.path.join(VAL_LOCALE, "previsioni"), dirs_exist_ok=True)

pesi_arg = " ".join(f'"{n}={p}"' for n, p in PESI.items())
%cd {WORK_DIR}
!python tracknet_finetune/valuta_tracknet.py --etichette "{ETICHETTE}" --video_dir inputs --pesi {pesi_arg} --uscita {VAL_LOCALE}

os.makedirs(VAL_DRIVE, exist_ok=True)
for f in glob.glob(os.path.join(VAL_LOCALE, "voto.*")) + glob.glob(os.path.join(VAL_LOCALE, "confronto_*.csv")):
    shutil.copy(f, VAL_DRIVE)
shutil.copytree(os.path.join(VAL_LOCALE, "previsioni"), os.path.join(VAL_DRIVE, "previsioni"), dirs_exist_ok=True)
display(Markdown(open(os.path.join(VAL_LOCALE, "voto.md"), encoding="utf-8").read()))
print("Copia dei risultati su Drive:", VAL_DRIVE)

## Dopo

- Se il voto migliora (soprattutto **richiamo** e **F1** nei gruppi *vicino al colpo* e *pallina mossa*, senza aumentare i **FP2** nel gruppo *a caso*): integrazione nel programma, con il confronto A/B sui video di riferimento.
- Se migliora poco: secondo fine-tuning con i nostri video (etichettati come il set di test, ma su video **diversi** da quelli del test).